In [2]:
import pennylane as qp
import numpy as np

In [ ]:
n_bits = 4
dev = qp.device("default.qubit", wires=n_bits)

def multisol_oracle_matrix(combos):
    """Return the oracle matrix for a set of solutions.

    Args:
        combos (list[list[int]]): A list of secret bit strings.

    Returns:
        array[float]: The matrix representation of the oracle.
    """
    indices = [np.ravel_multi_index(combo, [2]*len(combo)) for combo in combos]

    my_array = np.identity(2**len(combos[0])) # Create the identity matrix
    for index in indices:
        my_array[index, index] = -1

    return my_array

@qp.qnode(dev)
def multisol_pair_circuit(x_tilde, combos):
    """Implements the circuit for testing a pair of combinations labelled by x_tilde.
    
    Args:
        x_tilde (list[int]): An (n_bits - 1)-bit string labelling the pair to test.
        combos (list[list[int]]): A list of secret bit strings.

    Returns:
        array[float]: Probabilities on the last qubit.
    """
    for i in range(n_bits-1): # Initialize x_tilde part of state
        if x_tilde[i] == 1:
            qp.PauliX(wires=i)
    
    qp.H(n_bits-1)
    qp.QubitUnitary(multisol_oracle_matrix(combos), wires=range(n_bits))
    qp.H(n_bits-1)
    
    return qp.probs(wires=n_bits-1)


In [7]:
def parity_checker(combos):
    """Use multisol_pair_circuit to determine the parity of a solution set.

    Args:
        combos (list[list[int]]): A list of secret combinations.

    Returns: 
        int: The parity of the solution set.
    """
    parity = 0
    x_tilde_strs = [np.binary_repr(n, n_bits-1) for n in range(2**(n_bits-1))]
    x_tildes = [[int(s) for s in x_tilde_str] for x_tilde_str in x_tilde_strs]
    for x_tilde in x_tildes:
        # IMPLEMENT PARITY COUNTING ALGORITHM
        prob = multisol_pair_circuit(x_tilde, combos)
        parity += round(prob[1].item())

    return parity % 2